# DeepfakeDetector — Colab Training

Before running: in Colab go to **Runtime → Change runtime type → T4 GPU** (or any available GPU).

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/Hanineyasser/DeepfakeDetector.git"
REPO_DIR = Path("/content/DeepfakeDetector")

if not Path("convnext_trainer.py").exists():
    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
else:
    !git pull

print("Working directory:", os.getcwd())

In [ ]:
from pathlib import Path
import shutil

from google.colab import drive
drive.mount("/content/drive")

for local_name, drive_name in [
    ("models", "DeepfakeDetector_models"),
    ("lightning_logs", "DeepfakeDetector_logs"),
]:
    drive_dir = Path("/content/drive/MyDrive") / drive_name
    drive_dir.mkdir(parents=True, exist_ok=True)

    local_dir = Path(local_name)
    if local_dir.is_symlink():
        local_dir.unlink()
    elif local_dir.exists():
        shutil.rmtree(local_dir)

    local_dir.symlink_to(drive_dir)
    print(f"{local_name}/ -> {drive_dir}")

In [ ]:
!pip install -q pytorch-lightning kagglehub

In [ ]:
import os
from pathlib import Path
import kagglehub
import yaml

# Add any Kaggle real/fake face datasets here (each should have train/val folders with real & fake subfolders)
KAGGLE_DATASETS = [
    "xhlulu/140k-real-and-fake-faces",        # StyleGAN vs real faces
    "manjilkarki/deepfake-and-real-images",   # Deepfake/face-swap vs real faces
    "ayushmandatta1/deepdetect-2025",         # Diffusion (SD3, Midjourney, DALL-E 3) + StyleGAN3
]
RESUME_TRAINING = False  # Set True only to continue an interrupted multi-dataset run


def find_split_dirs(root: Path):
    train_dirs, val_dirs, test_dirs = [], [], []
    for dirpath, dirnames, _ in os.walk(root):
        lower_names = {d.lower() for d in dirnames}
        if "real" in lower_names and "fake" in lower_names:
            p = Path(dirpath)
            name = p.name.lower()
            if "train" in name:
                train_dirs.append(str(p))
            elif any(k in name for k in ("val", "valid", "validation")):
                val_dirs.append(str(p))
            elif "test" in name:
                test_dirs.append(str(p))
    return train_dirs, val_dirs or test_dirs


train_paths, val_paths = [], []
for slug in KAGGLE_DATASETS:
    ds_root = Path(kagglehub.dataset_download(slug))
    t_dirs, v_dirs = find_split_dirs(ds_root)
    print(f"{slug} -> train: {t_dirs}, val: {v_dirs}")
    train_paths.extend(t_dirs)
    val_paths.extend(v_dirs)

with open("config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

cfg["train_paths"] = train_paths
cfg["val_paths"] = val_paths
cfg["batch_size"] = 64
cfg["num_workers"] = min(4, os.cpu_count() or 2)

last_ckpt = Path("models/last.ckpt")
cfg["resume_from_checkpoint"] = str(last_ckpt) if (RESUME_TRAINING and last_ckpt.exists()) else None

with open("config.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Total train_paths:", len(train_paths), "| Total val_paths:", len(val_paths))
print("Batch size:", cfg["batch_size"], "| num_workers:", cfg["num_workers"])
print("Resume from checkpoint:", cfg["resume_from_checkpoint"])

In [ ]:
import torch
from convnext_trainer import main

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

main()